# A full business solution

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

## Setup

Key imports and why each is needed:

| Import | Purpose |
|---|---|
| `openai.OpenAI` | OpenAI Python SDK for calling GPT models via the Responses API |
| `pydantic.BaseModel` | Define typed schemas for Structured Outputs — guarantees model responses match an expected shape |
| `dotenv.load_dotenv` | Load `OPENAI_API_KEY` from a `.env` file without hard-coding credentials |
| `IPython.display` | Render and live-update Markdown in the notebook output cell |
| `scraper` | Local module — wraps Playwright to fetch page content and links from JavaScript-rendered sites |
| `nest_asyncio` | Patches the running event loop so `asyncio` works inside Jupyter's own async environment |

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import playwright_fetch_website_contents, playwright_fetch_website_links
from pydantic import BaseModel
from openai import OpenAI

import nest_asyncio
nest_asyncio.apply() 


## Configuration

Validates `OPENAI_API_KEY` from `.env` and initialises the OpenAI client.

Two models are used in this pipeline for different reasons:

| Model | Used for | Rationale |
|---|---|---|
| `gpt-5-nano` | Link classification | Fast and cheap — the task is simple classification, not prose generation |
| `gpt-4.1-mini` | Brochure generation | Higher quality output needed for polished, audience-aware marketing copy |

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


## Using the Playwright Implementation

Playwright launches a real Chromium browser to fully execute JavaScript before extracting content. This is essential for modern sites that rely on client-side rendering — plain HTTP clients like `requests` would miss most of the links and text.

### Step 0 — Fetch all raw links from the landing page

`playwright_fetch_website_links(url)` returns a deduplicated list of absolute URLs found on the page. Many of these will be irrelevant to a brochure (login pages, dataset listings, individual model cards, etc.) — that filtering happens in the next step.

In [3]:
url = "https://huggingface.co"
hf_links = playwright_fetch_website_links(url)
len(hf_links)

57

In [4]:
hf_links[:5]

['https://huggingface.co/',
 'https://huggingface.co/models',
 'https://huggingface.co/datasets',
 'https://huggingface.co/spaces',
 'https://huggingface.co/docs']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

### Building the User Prompt

`get_links_user_prompt(url)` fetches all raw links from the page and appends them to the instruction text, producing the full user message to send alongside the system prompt.

Separating the static *instruction* (system prompt) from the dynamic *data* (user prompt) keeps each part maintainable independently — the system prompt never needs to change, only the data does.

The cell below previews the exact string that will be sent as the user message, which is useful for debugging prompt content before making an API call.

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = playwright_fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt(url))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://huggingface.co/
https://huggingface.co/models
https://huggingface.co/datasets
https://huggingface.co/spaces
https://huggingface.co/docs
https://huggingface.co/enterprise
https://huggingface.co/pricing
https://huggingface.co/login
https://huggingface.co/join
https://huggingface.co/blog/ggml-joins-hf
https://huggingface.co/Qwen/Qwen3.5-397B-A17B
https://huggingface.co/zai-org/GLM-5
https://huggingface.co/MiniMaxAI/MiniMax-M2.5
https://huggingface.co/nvidia/personaplex-7b-v1
https://huggingface.co/Nanbeige/Nanbeige4.1-3B
https://huggingface.co/spaces/itsMarco-G/reachy_phone_home
https://huggingface.co/spaces/Wan-AI/Wan2.2-Animate
https://huggingface.co/spaces/r3gm/wan2-2-fp8da-aoti-preview

In [8]:
MODEL = "gpt-5-nano"

class Link(BaseModel):
    type: str
    url: str

class PageLinks(BaseModel):
    links: list[Link]


def select_relevant_links(url):
    response = openai.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        # text_format={"type": "json_object"}
        text_format=PageLinks
    )
    result = response.output_parsed
    # links = json.loads(result)
    links = result.model_dump()
    print(f"Found {len(links['links'])} relevant links")
    return links

### Structured Outputs with Pydantic

`select_relevant_links` uses `openai.responses.parse()` with a **Pydantic model** as `text_format`. This is OpenAI's Structured Outputs feature — the model is constrained to return JSON that exactly matches the schema, so no manual parsing or error handling is needed.

The schema mirrors the expected shape:

```python
class Link(BaseModel):
    type: str   # e.g. "about page", "careers page"
    url:  str   # absolute URL

class PageLinks(BaseModel):
    links: list[Link]
```

`.model_dump()` converts the parsed Pydantic object back to a plain Python dict for easier downstream use.

In [9]:
result = select_relevant_links(url)

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10a249e80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10a249e80> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asy

Found 6 relevant links


In [10]:
result

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'product overview page', 'url': 'https://huggingface.co/models'}]}

## Second Step: Build the Brochure

With the relevant links identified, the pipeline now:

1. Fetches the **full rendered text** of the landing page and each relevant sub-page using Playwright
2. Assembles all scraped content into a single user prompt
3. Sends it to `gpt-4.1-mini` to generate a polished Markdown brochure

`fetch_page_and_all_relevant_links` handles steps 1–2. `get_brochure_user_prompt` wraps the result into the final prompt. `create_brochure` ties everything together and renders the output.

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = playwright_fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += playwright_fetch_website_contents(link["url"])
    return result

In [12]:
page_relevant_links = fetch_page_and_all_relevant_links(url)

Found 6 relevant links


In [13]:
print(page_relevant_links)

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending onthis week
Models
Qwen/Qwen3.5-397B-A17B
Updated 1 day ago
•
133k
•
784
zai-org/GLM-5
Updated 8 days ago
•
177k
•
1.39k
MiniMaxAI/MiniMax-M2.5
Updated 5 days ago
•
173k
•
812
nvidia/personaplex-7b-v1
Updated 6 days ago
•
539k
•
2.1k
Nanbeige/Nanbeige4.1-3B
Updated 2 days ago
•
130k
•
633
Browse 2M+ models
Spaces
Reachy Phone Home
📱
311
Phone focus companion for Reachy Mini
Wan2.2 Animate
👁
4.77k
Wan2.2 Animate
Wan2.2 14B Preview
🐌
829
generate a video from an image with a text prompt
Z Image Turbo
🖼
2.25k
Generate high-quality images from t

In [14]:
len(page_relevant_links)

25559

### Brochure Prompt Design

The **system prompt** instructs the model to write a short, audience-aware brochure targeting three distinct reader types: prospective **customers**, **investors**, and **recruits**. This directs the model to surface culture, product value, and careers content — rather than generating generic marketing copy.

The **user prompt** (`get_brochure_user_prompt`) injects all the scraped content, structured as:

```
## Landing Page:
<full page text>

## Relevant Links:

### Link: about page
<scraped text>

### Link: careers page
<scraped text>
...
```

The next cell prints the assembled prompt so you can inspect what content was scraped before committing to an API call.

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt
    return user_prompt

In [17]:
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10a249e80> is already entered


Found 31 relevant links


Task was destroyed but it is pending!
task: <Task pending name='Task-158' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-160' coro=<Kernel.shell_main() running at /Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py:590> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/patrickwalukagga/Projects/personal/llm_engineering/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/patrickwalukagga/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/json/decoder.py:354: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)
Task was destroyed but it is pending!
task: <Task pending name='Task-160' coro=<Kernel.shell_main() running at /Users/patrickw


You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Pricing
Log In
Sign Up
NEW
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
Get started with Inference in seconds 🚀
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending onthis week
Models
Qwen/Qwen3.5-397B-A17B
Updated 1 day ago
•
133k
•
784
zai-org/GLM-5
Updated 8 days ago
•
177k
•
1.39k
MiniMaxAI/MiniMax-M2.5
Updated 5 days ago
•
173k
•
812
nvidia/personaplex-7b-v1
Updated 6 days ago
•
539k
•
2.1k
Nanbeige/Nanbeige4.1-3B
Updated 2 days ago
•
130k
•
633
Browse 2M+ models
Spaces
Reachy Phone Home

In [19]:
def create_brochure(company_name, url):
    brochure_user_prompt = get_brochure_user_prompt(company_name, url)
    response = openai.responses.create(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt}
        ],
    )
    result = response.output_text
    display(Markdown(result))

### Generate Brochures

Call `create_brochure(company_name, url)` with any company name and website URL. The full pipeline runs end-to-end — scraping, link selection, content assembly, and generation — and renders the result as formatted Markdown inline.

> **Note:** Each call makes multiple Playwright browser sessions (one for links, one per relevant page) and several API calls, so it may take 20–60 seconds depending on the site and number of relevant links found.

In [20]:
company_name = "OpenAI"
url = "https://openai.com"
create_brochure(company_name, url)

Found 21 relevant links


# OpenAI: Pioneering Artificial General Intelligence for Humanity

---

## About OpenAI

OpenAI is a leading AI research and deployment company founded in 2015. Its core mission is to **ensure that artificial general intelligence (AGI)**—highly autonomous systems that outperform humans at most economically valuable work—**benefits all of humanity**.

Operating under a unique structure, OpenAI consists of a nonprofit arm, the **OpenAI Foundation**, and a for-profit entity, the OpenAI Group (a public benefit corporation). Together, they govern and accelerate AI advances responsibly and ethically.

---

## Our Mission & Values

- **Humanity First**: AI is built to elevate people and society, prioritizing broad societal benefits.
- **Act with Humility**: We recognize the limits of knowledge and remain open to new ideas and feedback.
- **Feel the AGI**: We respect AGI’s transformative power and manage its risks with rigor.
- **Ship Joy**: We create optimistic, impactful AI products that enhance lives.

Operating Principles:

- **Find a Way**: Empowering individuals and teams to solve important problems creatively.
- **Creativity over Control**: Foster innovation with flexibility and principles-driven problem solving.
- **Update Quickly**: Adapt rapidly with new information for continuous improvement.
- **Intense Focus**: Maintain clarity and resilience to deliver global impact.

---

## Cutting-Edge Products & Research

- **GPT-5 Series**: Our smartest, fastest AI language models with advanced reasoning and multimodal capabilities (handling text, image, audio, and video).
- **Codex**: AI that accelerates software engineering by writing, reviewing, and debugging code.
- **Sora**: A groundbreaking app for generating hyperrealistic videos with sound and dynamic motion.
- **OpenAI Frontier**: An enterprise platform powering AI agents that operate securely at scale, integrate with corporate systems, and improve over time.
- **Open Models**: Open-weight, customizable reasoning models for deployment anywhere.
- **Whisper & Voice Agents**: Advanced speech recognition and natural voice synthesis technologies.
- **Safety & Alignment**: Continual safety research ensures AI systems act with ethical and social responsibility.

Our research publications push the boundaries of AI understanding, advancing theory and real-world applications, including biology, medicine, and economics.

---

## Customers & Industry Impact

OpenAI collaborates with thousands of customers, including:

- **Enterprises**: Boost productivity and automate critical workflows with ChatGPT Business and the API Platform.
- **Healthcare & Life Sciences**: Accelerate research and improve patient outcomes.
- **Financial Services & Banking**: Scale AI-native operations securely.
- **Manufacturing & Energy**: Optimize capacity and mitigate natural disaster impacts.
- **Retail & Communications**: Enhance customer experiences and operational efficiency.

Our solutions enable sector-specific AI adoption with enterprise-grade **security**, **compliance** (GDPR, CCPA, HIPAA), and **privacy protections** like no customer data usage for training.

---

## Careers at OpenAI

OpenAI seeks **curious, passionate, and talented individuals** from diverse disciplines committed to building safe and beneficial AI:

- Roles across AI research, engineering, policy, safety, product management, and more.
- Values-centered culture prioritizing humanity, humility, creativity, focus, and continuous learning.
- Benefits include comprehensive health coverage, mental healthcare support, paid parental leave, fertility and family planning coverage, generous paid time off, and learning stipends.
- Programs such as the **OpenAI Residency** provide pathways for researchers and engineers new to AI.
- Employees enjoy a culture of optimism, responsibility, and impactful work that surpasses previous technological breakthroughs.

Join OpenAI in shaping the future of AI for a better world.

[View current openings](https://openai.com/careers)

---

## Brand & Identity

OpenAI’s brand reflects the harmony of **human-centered warmth and technological precision**:

- Logo and wordmark symbolize the intersection of humanity and technology.
- Unique typeface, **OpenAI Sans**, merges geometric clarity with friendliness.
- Branding is carefully guided to maintain consistency and proper partnership use.

---

## Getting Started & Partnering

- **For Developers**: Access the API Platform with flexible pricing and powerful models optimized for varied workloads and multimodal inputs.
- **For Businesses**: Leverage ChatGPT Business and OpenAI Frontier to deploy AI-powered agents and workflows tailored to enterprise needs.
- **For Startups**: Benefit from specialized tools, learning resources, and community support designed for ambitious innovators.
- **Enterprise-level Security**: Role-based access control, encryption, audit logs, and compliance certifications ensure data privacy and governance.

Gain expert guidance and partner with OpenAI’s solutions architects for effective AI integration.

---

## Contact & Learn More

Explore OpenAI’s research, products, documentation, and more at [openai.com](https://openai.com)  
Email press inquiries: partnercomms@openai.com  
Email legal inquiries: legal@openai.com

---

### OpenAI — Advancing AGI to benefit all of humanity. Join us on this transformational journey.

In [21]:
company_name = "Sunbird AI"
url = "https://sunbird.ai"
create_brochure(company_name, url)

Found 13 relevant links


# Sunbird AI Brochure

---

## Who We Are
Sunbird AI is a pioneering artificial intelligence research organization dedicated to developing AI technologies and solutions tailored to the unique challenges and opportunities in Africa. Headquartered in Kampala, Uganda, Sunbird AI combines cutting-edge AI research with social impact initiatives aimed at improving lives across the continent.

---

## Our Mission
To harness artificial intelligence for African problems, driving social good through scalable, adaptable, and inclusive technology solutions that empower local communities, enhance government services, promote sustainable development, and preserve cultural and linguistic diversity.

---

## Flagship Projects & Innovations

### Sunflower Multilingual Assistant
- A state-of-the-art multilingual AI assistant supporting **31 Ugandan languages**.
- Provides accurate translation, summaries, and explanations.
- Enables natural, conversational interaction in local languages.
- Available online and via API for developers.

### African Language Technology
- Development of Natural Language Processing (NLP) tools for local languages such as Acholi, Ateso, Luganda, Lugbara, and Runyankole.
- Created SALT dataset - 25,000 sentences translated across six Ugandan languages on topics like healthcare, agriculture, and society.
- Advanced speech technology including pioneering Text-to-Speech (TTS) for Luganda.
- Projects enhancing citizen engagement through voice feedback systems supporting multiple local languages (e.g., SEMA Uganda, TRAC FM).

### Green Electrification Planning
- AI-driven site identification system for renewable energy in Uganda.
- Partnerships with GIZ and the Ugandan Ministry of Energy to increase electricity access.
- Use of satellite and remote sensing data to optimize electrification strategies, particularly in underserved rural regions.

### Environmental Sensing & Noise Pollution
- Real-time acoustic monitoring of Kampala and Entebbe to assess urban noise pollution and its health impacts.
- Collaborations with city authorities to provide actionable data for urban planning and public health.

### Citizen Feedback & Social Impact
- AI-powered voice transcription and translation to amplify marginalized voices in public discourse.
- Support for government initiatives like Uganda's Parish Development Model to improve rural communication and service delivery.

---

## Our Customers & Partners
- **Government Ministries:** Ugandan Ministry of Energy, Ministry of ICT and National Guidance.
- **International Organizations:** German Agency for International Cooperation (GIZ), United Nations Pulse Lab.
- **Research Institutes:** Makerere University AI Lab, Initiative Prospective Agricole et Rurale (IPAR), Center for the Study of African Economies (CSEA).
- **Community Organizations:** SEMA Uganda, TRAC FM radio.

We also collaborate in consortia focusing on maternal and sexual reproductive health, gender equality, and inclusive AI development in Sub-Saharan Africa.

---

## Our Team & Culture
Sunbird AI is driven by a diverse, multidisciplinary team of experts passionate about applying AI for sustainable development and social good in Africa.  
- **Leadership:** Experienced directors and senior researchers with strong academia-industry linkages, including from Makerere University and global institutions like Google AI.
- **Researchers and Engineers:** Software engineers, machine learning specialists, data scientists, and operations managers deeply embedded in African contexts.
- **Community Engagement:** A focus on inclusive communication strategies and public participation in AI-related initiatives.

We foster a collaborative culture that encourages innovation, lifelong learning, and alignment with societal impact goals. Our community embraces openness—data, models, and code are made freely accessible to accelerate innovation.

---

## Careers and Fellowship Opportunities
- **Sunbird AI Fellows Program:** A competitive 6-month fellowship enabling researchers and practitioners to work on impactful AI projects addressing African challenges.
- Fellowship projects can be full-time or part-time and conducted remotely or at our Kampala office.
- Perfect for PhD students and early career professionals interested in real-world AI applications.
- Fellows receive mentorship from Sunbird AI staff and opportunities to contribute to open-source tools and datasets.
- Applications are open yearly; interested candidates should prepare a motivation letter, project plan aligned with Sunbird’s mission, and optionally share work samples.

---

## Get Involved & Contact Us
- Book 30-minute advisory sessions to explore AI in your organizational context.
- Engage with our open projects or apply for a fellowship.
- Collaborate with us to build AI that uplifts African societies.

**Contact Information:**  
Plot 15, Naguru East Road, Kampala, Uganda  
PO Box 11296 Kampala, Uganda  
Email: info@sunbird.ai  

Follow us on Twitter, Youtube, and Medium for latest updates.  

---

Sunbird AI | Artificial Intelligence Research for African Solutions  
**Empowering Africa through Intelligent Innovation**

## Streaming Variant: `stream_brochure`

A small but impactful improvement over `create_brochure` — instead of waiting for the full response before displaying anything, tokens are streamed back and the Markdown output is updated in place as they arrive, giving the familiar typewriter effect.

**How it works:**

- `stream=True` activates the streaming Responses API
- Each `response.output_text.delta` event carries the next text chunk
- `update_display(Markdown(response), display_id=...)` re-renders the growing Markdown string in the same output cell — no flicker, no duplicate cells

Use `stream_brochure` in preference to `create_brochure` for interactive use; use `create_brochure` when you need the full string returned (e.g. for saving to a file).

In [22]:
def stream_brochure(company_name, url):
    brochure_user_prompt = get_brochure_user_prompt(company_name, url)
    stream = openai.responses.create(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for event in stream:
        if event.type == "response.output_text.delta":
            response += event.delta
            update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
company_name = "HuggingFace"
url = "https://huggingface.co"
stream_brochure(company_name, url)

Found 9 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community and collaboration platform transforming the future of machine learning. Serving as a central hub, it empowers machine learning engineers, scientists, and users worldwide to share, explore, and develop open-source ML models, datasets, and applications. Known as the “Home of Machine Learning,” Hugging Face fosters an open and ethical AI future through community collaboration and cutting-edge technology innovation.

---

## What We Offer

### Platform Features
- **Model and Dataset Hub**: Access over 2 million models and 500,000+ datasets spanning all machine learning modalities — text, image, audio, video, and even 3D.
- **Spaces**: A unique AI app directory allowing users to host and share ML applications and demos easily.
- **Open Source Tools**: Popular libraries like Transformers (156k+ stars), Diffusers for diffusion models, Tokenizers, PEFT for efficient fine-tuning, and many more.
- **Inference API**: Access to 45,000+ models from leading AI providers with no service fees via a unified API.
- **Compute Solutions**: Optimized inference endpoints and scalable GPU options starting as low as $0.60/hour, plus free community GPU grants for side projects.

---

## Enterprise Solutions

Hugging Face provides advanced AI platforms with enterprise-grade security, access controls, dedicated support, and flexible contract options starting at $50/user/month. Features include:
- Single Sign-On (SSO) and SAML support
- Data residency with storage regions
- Granular access control with resource groups
- Audit logs and detailed analytics dashboards
- Increased compute scalability with options like ZeroGPU Quota Boost
- Private datasets viewing and storage
- Supported by priority customer support and managed billing

Trusted by over 50,000 organizations, including tech giants like Google, Microsoft, Amazon, Meta, NVIDIA, Salesforce, IBM, OpenAI, and many others, Hugging Face powers innovations across industries worldwide.

---

## Community & Culture

Hugging Face thrives on a vibrant and growing community committed to open-source collaboration, ethical AI development, and innovation. The platform encourages:
- Sharing research and ML projects openly to build an inclusive AI ecosystem
- Learning through community blogs, forums, and numerous educational courses on large language models, robotics, reinforcement learning, diffusion models, audio and vision ML, and more
- Experimentation and portfolio building with model and dataset hosting, Spaces apps, and public collaboration tools
- Continuous engagement through open discussions, joint projects, and support channels on Discord, GitHub, and other social platforms

The company ethos centers on transparency, inclusivity, rapid iteration, and the empowerment of developers and researchers worldwide to shape the future of AI.

---

## Careers at Hugging Face

Joining Hugging Face means becoming part of the heart of the AI revolution. The company seeks talented individuals passionate about open-source AI, community-building, and pushing the boundaries of machine learning technology. Career opportunities span a variety of fields — engineering, research science, product, community management, and more.

Employees benefit from:
- Cutting-edge projects with high-impact technology
- A collaborative and supportive international work culture
- An environment fostering learning, growth, and innovation
- Direct engagement with top AI researchers and practitioners globally

Explore open positions and contribute toward building the future AI ecosystem.

---

## Pricing Overview

- **Free:** Access to the Hugging Face Hub for exploration, collaboration, and sharing models and datasets.
- **PRO Account ($9/month):** Enhanced private storage, inference credits, priority queue access, and Spaces development features.
- **Team Plan ($20/user/month):** Enterprise-grade features like SSO, audit logs, granular access, analytics, and advanced compute options.
- **Enterprise Plan (from $50/user/month):** Customized onboarding, highest resource limits, advanced security, legal compliance, personalized support.

Additional per-TB pricing for large-scale data storage and on-demand hardware upgrades for Spaces applications.

---

## Join Us in Building the Future of AI!

Create, collaborate, and innovate with Hugging Face — where the machine learning community builds the AI of tomorrow.

Explore Models: https://huggingface.co/models  
Discover Datasets: https://huggingface.co/datasets  
Join the Community: https://huggingface.co/spaces  
Careers & More: https://huggingface.co/careers  

---

**Hugging Face**  
The AI community building the future.  
#BuildTheFutureTogether #OpenAI #MachineLearning

In [24]:
company_name = "OpenAI"
url = "https://openai.com"
stream_brochure(company_name, url)

Found 14 relevant links


# OpenAI Brochure

---

## About OpenAI

OpenAI is a leading AI research and deployment company committed to ensuring that artificial general intelligence (AGI)—highly autonomous systems surpassing human capability in most economically valuable work—benefits all of humanity. Founded in 2015, OpenAI is structured as a public benefit corporation governed by the nonprofit OpenAI Foundation, combining the agility of a startup with stewardship of a philanthropic mission.

---

## Mission & Vision

- **Mission:** To create safe and beneficial AGI or to aid others in achieving this outcome.
- **Vision:** To ensure AGI brings broad, distributed benefits to society and to minimize harmful uses or undue power concentration.
- **Charter Highlights:**  
  - Commit to long-term safety and cooperation with global stakeholders.  
  - Prioritize humanity's interests as the core fiduciary duty.  
  - Maintain technical leadership with transparency and responsibility.

---

## Products & Research Highlights

- **GPT-5 Series:** The smartest, fastest, and most useful AI models that generate, understand, and reason across text, voice, image, and vision domains.
- **Codex:** AI coding assistants to build, debug, and ship software faster.
- **Sora:** Multimodal models delivering photorealistic and content-rich outputs.
- **OpenAI Frontier:** Enterprise AI platform powering autonomous AI coworkers and workflow automation with integrated security and governance.
- **Safety Research:** Continuous innovations to align AI behavior with human values, prevent misuse, and address emerging risks.

---

## Business and Customer Solutions

OpenAI serves thousands of companies with versatile AI platforms and products:

- **ChatGPT for Business & Enterprise:**  
  Unlimited access to advanced models, team collaboration tools, robust security, and app integrations (Google Drive, SharePoint, GitHub, etc.)
  
- **API Platform:**  
  Fastest, most powerful API platform for developing AI applications handling text, image, audio, and multi-modal inputs.

- **Industry Applications:**  
  - Financial services: trust and market advantage  
  - Healthcare & Life Sciences: improved diagnostics, drug approvals  
  - Manufacturing & Energy: operational optimizations and disaster mitigation  
  - Communications & Retail: customer support and personalized experiences

- **Enterprise Trust & Governance:**  
  Comprehensive privacy controls, auditability, identity and access management, and compliance with SOC 2, ISO standards, HIPAA support, GDPR and CCPA.

---

## Company Culture & Careers

OpenAI fosters a passionate, inclusive, and resilient work environment grounded in a set of core values and operating principles:

- **Key Values:**  
  - **Humanity First:** AI to elevate people and society.  
  - **Humility:** Open, iterative learning and adaptation.  
  - **Responsibility:** Careful development acknowledging AI’s transformative power.  
  - **Joy:** Development of joyful, impactful products.

- **Operating Principles:**  
  - Find creative solutions over control.  
  - Update quickly based on new insights.  
  - Intense focus to create impactful change.

- **Perks & Benefits:**  
  Health, dental, vision, mental health services, global travel insurance, paid parental leave, fertility and family planning coverage, generous PTO, learning stipends, community resource groups, and regular meals.

- **Professional Growth:**  
  OpenAI Residency - a 6-month program for engineers and researchers entering AI.

- **Diversity in Backgrounds:**  
  OpenAI seeks collaborators with varied expertise and perspectives, welcoming early-career talent, interns, and seasoned professionals.

---

## OpenAI Foundation

- Holds equity valued at ~$130 billion, one of the best-resourced global philanthropic organizations.
- Focuses on accelerating health breakthroughs globally and AI resilience solutions.
- Supports nonprofits through the People-First AI Fund with initial $50 million funding for AI innovation aligned with public good.

---

## Safety & Responsibility

- Dedicated to making AI safe for everyone by embedding safety at every development stage.
- Approaches include teaching AI ethics, rigorous testing, real-world feedback loops, and collaboration with experts and policymakers.
- Develops safeguards against harmful content, bias, misinformation, privacy violations, and misuse.
- Publishes safety research and detailed system cards for transparency about risks and mitigations.

---

## Brand & Partnerships

- Distinctive identity built around clarity, inclusivity, and technological precision.
- Logos and marks use strict guidelines emphasizing consistency and respect for intellectual property.
- Partnerships are managed carefully to maintain brand integrity and ensure positive association.

---

## Join OpenAI

Be part of a transformational mission shaping the future of technology and society.

- Explore careers at [OpenAI Careers](https://openai.com/careers)  
- Engage with startups and builders via [OpenAI for Startups](https://openai.com/startups)  
- Access community learning, live events, and guided resources

---

## Contact & Resources

- Explore AI products and solutions on [OpenAI.com](https://openai.com)  
- Access research publications and latest advancements  
- Get support and developer documentation  
- Follow OpenAI news, stories, and podcasts for ongoing innovation updates

---

**OpenAI — Building safe and beneficial artificial intelligence to uplift all of humanity.**

In [25]:
company_name = "Sunbird AI"
url = "https://sunbird.ai"
stream_brochure(company_name, url)

Found 17 relevant links


# Sunbird AI Brochure

---

## Who We Are

Sunbird AI is an innovative artificial intelligence research organization dedicated to applying cutting-edge AI technologies to solve Africa’s unique challenges. Based in Kampala, Uganda, our mission is to create scalable, adaptable AI solutions that significantly impact society by addressing local problems in health, environment, agriculture, language access, energy, and social development.

---

## Our Culture

At Sunbird AI, we are driven by a strong commitment to using computational intelligence for social good. We foster a collaborative and dynamic research environment where academia, industry experts, and communities work hand-in-hand. Our team is passionate about lifelong learning, innovation, and bridging the gap between AI research and practical applications tailored to African contexts.

Our culture values:

- **Social Impact**: Prioritizing AI solutions that improve livelihoods and drive sustainable development.
- **Inclusiveness**: Building technologies that respect and uplift local languages and cultures.
- **Collaboration**: Strong partnerships with universities, governmental agencies, NGOs, and industry leaders.
- **Transparency**: Open data, code, and models freely available to support growth beyond our organization.

---

## Our Team

We are proud of our diverse, highly skilled team blending academia and industry experience:

- **Engineer Bainomugisha, Director & Founder** — A leader in social impact and AI education with 15+ years of experience spearheading projects that improve African cities' environmental health.
- **Ernest Mwebaze, Executive Director** — AI research and practical application expert with prior roles at Google AI and United Nations Pulse labs.
- **John Quinn, Director** — AI veteran with nearly two decades of experience working in Africa, blending satellite imagery analysis and community-driven language technology initiatives.
- **Software Engineers, Data Scientists, Researchers, and Fellows** — Talented professionals and upcoming AI fellows collaborating to develop scalable and impactful AI solutions.

For a full team overview and board members, including experts in finance, project management, communications, and embedded systems engineering, please contact us.

---

## Flagship Projects & Solutions

**Sunflower Multilingual Assistant**

- Accurate translation, summaries, and conversational AI across 31 Ugandan languages.
- Enables access to information for hundreds of millions of local language speakers.
- Open API and freely available resources to encourage further innovations.

**African Language Technology**

- Development of multilingual datasets and neural translation models for local languages such as Luganda, Acholi, Ateso, and others.
- Creation of the first ever text-to-speech models for Ugandan languages using crowdsourced voices.
- Collaboration with universities to empower communities and bridge linguistic barriers.

**Green Electrification Planning**

- AI-driven site identification system to optimize renewable energy deployment in Uganda, working with GIZ and the Ministry of Energy.
- Forecasting human settlement growth to inform future electrification strategies.
- Supports Uganda’s National Electrification Strategy to connect millions of households by 2030.

**Citizen Feedback Platforms**

- Voice-based and multi-lingual feedback systems deployed for healthcare and public services.
- Partnerships with Uganda’s Ministry of ICT, SEMA, and community radio for inclusive citizen engagement.
- AI models transcribe, translate, and summarize citizen feedback to inform policy and service delivery.

**Environmental Sensing & Noise Pollution**

- Real-time acoustic monitoring to assess urban noise pollution in Kampala and Entebbe.
- Custom embedded hardware collecting data to support public health interventions.
- Collaborations with Entebbe City Authority and Kampala Capital City Authority for better urban planning.

---

## Our Customers and Partners

- **Government Ministries:** Ministry of Energy and Ministry of ICT, Uganda.
- **International Agencies:** German Agency for International Cooperation (GIZ), United Nations.
- **Research Institutions:** Makerere University AI Lab, Google AI research labs in Africa.
- **Community Organizations:** SEMA Uganda, Radio TRAC FM.
- **Regional Networks:** GRAIN Hub for gender equality in Sub-Saharan Africa, IPAR Senegal, CSEA Nigeria.
- **Technology Partners:** Mozilla Common Voice for speech data.

---

## Careers at Sunbird AI

**Fellows Program**

- A competitive 6-month fellowship designed for research-oriented professionals and PhD students.
- Opportunity to work on impactful AI projects addressing African societal challenges.
- Flexible arrangements: full-time or part-time, online or on-site in Kampala.
- Fellows receive mentorship and can advance research aligned with their expertise and Sunbird’s mission.
- Applications are open year-round. Visit our Careers page or contact info@sunbird.ai for details.

**Hiring**

Sunbird AI seeks passionate individuals specializing in:

- Machine Learning & AI Research
- Software Engineering & Cloud Solutions
- Data Science & Geospatial Analysis
- Communications and Community Engagement
- Administration and Operations Management

Join a mission-driven team committed to equitable technology and African innovation.

---

## Get Involved

- **Join our Community:** Book a session with Sunbird AI experts for AI adoption consultations.
- **Explore Open Resources:** Access datasets, models, and APIs on African languages and other projects.
- **Partner with Us:** Collaborate on research, social impact initiatives, or capacity building.
- **Stay Informed:** Follow us on Twitter, Youtube, and Medium for updates and insights.

---

## Contact Us

Sunbird AI  
Plot 15, Naguru East Road  
Kampala, Uganda  
PO Box 11296 Kampala  

Email: info@sunbird.ai  
Twitter | YouTube | Medium

---

Sunbird AI – Harnessing Artificial Intelligence to Empower Africa’s Future

© 2025 All Rights Reserved

In [26]:
company_name = "Cardington Motors Uganda"
url = "https://cardingtonmotors.com"
stream_brochure(company_name, url)

Found 18 relevant links


# Cardington Motors Uganda  
*Your Premier Auto Garage in Kampala*  

---

## About Us  
At Cardington Motors, we are dedicated to making motor vehicle ownership and maintenance in Uganda less daunting, more affordable, and even a pleasure — especially as vehicles become increasingly sophisticated and computerized. Located at Kasirivu Close, Kisaasi Town Centre, Kampala, we offer fast, reliable, and professional vehicle care performed by ASE-certified technicians.  

Our vision is to completely satisfy our customers by keeping their vehicles safe, reliable, and operating at optimal performance. We pride ourselves on a friendly, helpful team that provides personalized service with professional standards, ensuring your vehicle is fixed right the first time.  

---

## Our Services  
We provide a comprehensive range of automotive services tailored to the needs of both foreign and domestic vehicle owners in Kampala. Our skilled professionals utilize state-of-the-art tools and diagnostic software to service your car efficiently and effectively.  

### Key Services:  
- **General Motor Services**  
  Routine maintenance including oil/filter changes, brake fluid flushes, belt inspections, coolant top-ups, and preventive checks to extend engine life, improve fuel economy, and avoid unexpected breakdowns.  
- **Computerised Diagnosis**  
  Advanced OBD-II scanners and manufacturer-level diagnostic tools help accurately identify faults and monitor live vehicle data, minimizing guesswork and unnecessary repairs.  
- **Body (Coach) Works**  
  Expert dent removal, panel beating, structural realignment, and corrosion-resistant paint refinishing to restore your vehicle's appearance and structural integrity.  
- **Wheel Alignment & Balancing**  
  Precise correction of camber, toe, and caster angles plus dynamic balancing to enhance handling, safety, and tire longevity.  
- **Engine and Gearbox Rebuild**  
  Full teardown, inspection, machining, and rebuilding of engines and transmissions with OEM tolerance standards to restore peak performance.  
- **Hybrid and Electric Vehicle Servicing**  
  Specialist care including high-voltage battery health checks, electric motor diagnostics, thermal management, and software updates to keep eco-friendly vehicles running at their best.  
- **Additional Services**  
  Motor vehicle valeting & detailing, appraisal & evaluation, car audio installation, air-conditioning recharge, parking sensor calibration, and battery repairs.  

---

## Why Choose Cardington Motors?  
- Every job is treated personally — you get dealership-quality workmanship in a friendly, welcoming atmosphere.  
- We invest in the latest specialist tools and diagnostic software tailored to your vehicle’s technology.  
- Our team only performs the necessary work, ensuring transparent pricing and no unnecessary repairs.  
- Certified to service a broad range of brands including Land Rover, Mercedes Benz, BMW, Audi, Volkswagen, Ford, Jeep, Volvo, Jaguar, Chrysler, Dodge, MINI, Porsche, Nissan, Toyota, and more.  
- Nearly all repairs and maintenance can be completed on the same day for your convenience.  

---

## Customer Testimonials  
> “Outstanding spray booth, cleanliness, friendly atmosphere, competitive prices, quality parts, and a ‘no problem’ attitude. Thank you so much!” — *Caroline Nalukenge*  

> “Fast, efficient, competitive, and comprehensive. Same day pick-up and delivery made it super convenient. Highly recommend!” — *Isaac Okoth*  

> “Exceptional service with great communication and attention to detail. Returned my car spotless. Truly go the extra mile.” — *Henry Sserwanda*  

---

## Company Culture  
We are a professional yet friendly team passionate about automotive expertise and customer satisfaction. Our culture fosters helpfulness, reliability, and ensuring the job is done right the first time. We embrace the latest automotive technologies including servicing of hybrid and electric vehicles, reflecting our commitment to innovation and sustainability.  

---

## Careers & Opportunities  
While our website does not currently list specific job openings, Cardington Motors seeks to grow its team with friendly, skilled professionals who are eager to work with a variety of vehicle models and modern diagnostic tools. Join us if you want to be part of a team that values quality workmanship, continuous learning, and customer care excellence.  

---

## Contact & Appointments  
**Phone:** (+256) 771 619 183 / (+256) 704 619 183 / (+256) 782 565 565  
**Email:** info@cardingtonmotors.com  
**Address:** Kasirivu Close, Kisaasi Town Centre, Kampala, P.O. Box 100070 Kampala GPO  
**Hours:**  
- Monday to Friday: 8:00am - 6:00pm  
- Saturday: 8:00am - 4:00pm  
- Sunday: Closed  

Book your service appointment online or call us anytime. Emergency assistance and 24/7 support available.  

---

Cardington Motors Uganda — **Where your vehicle gets the care it deserves.**  
Visit us at [cardingtonmotors.com](http://cardingtonmotors.com) to learn more or to book your next service today!